# rotation matrix (3-D, Y-axis) — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d-y-axis`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five rotation-matrix patterns that ramp from `cos/sin` → matrix assembly → rotate-a-vector → composition law → rotate-a-batch. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import math
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `rotation-matrix-3d-y-axis`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d-y-axis"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Y-axis rotation — quick refresher

**The matrix.** For a right-hand rotation by `θ` about the Y axis:
```
R_y(θ) = [[ cos θ,  0,  sin θ],
          [ 0,      1,  0    ],
          [-sin θ,  0,  cos θ]]
```

**Why the middle row is `[0, 1, 0]`.** The Y axis is the rotation axis — anything along Y stays where it is. The X-Z plane is what gets rotated.

**Right-hand convention.** Looking down the +Y axis, the rotation goes counter-clockwise: +X → -Z, +Z → +X.

**Acting on vectors.**
- Column-vector: `v' = R @ v` (input shape `(3,)`).
- Batch of row-vectors: `points' = points @ R.T` (input shape `(N, 3)`).

**Composition.** `R_y(α) @ R_y(β) = R_y(α + β)` — rotations about a single axis add angles.

### Exercise 1 — compute cos(θ) and sin(θ) as tensors

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall how to compute `cos(θ)` and `sin(θ)` as scalar tensors.
> Keywords: torch-cos, torch-sin, scalar-tensor
> ```

**KCs targeted:** `cos-sin-tensor-scalars`

Implement `ex1_cos_sin(theta)`. Given a 0-D tensor `theta` (an angle in radians), return a tuple `(cos_t, sin_t)` of two 0-D tensors.

Use `torch.cos` and `torch.sin`. Both inputs and outputs are torch tensors — don't convert to Python floats.

In [ ]:
def ex1_cos_sin(theta: Tensor) -> tuple:
    """Return (cos(theta), sin(theta)) as 0-D tensors."""
    raise NotImplementedError()


def _test_ex1():
    import math
    theta = t.tensor(math.pi / 2)
    c, s = ex1_cos_sin(theta)
    assert c.dim() == 0 and s.dim() == 0, 'both outputs must be 0-D'
    assert t.allclose(c, t.tensor(0.0), atol=1e-6), f'cos(pi/2) should be 0, got {c.item()}'
    assert t.allclose(s, t.tensor(1.0), atol=1e-6), f'sin(pi/2) should be 1, got {s.item()}'

    theta0 = t.tensor(0.0)
    c0, s0 = ex1_cos_sin(theta0)
    assert t.allclose(c0, t.tensor(1.0)) and t.allclose(s0, t.tensor(0.0)), 'cos(0)=1, sin(0)=0'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_cos_sin(theta: Tensor) -> tuple:
    return t.cos(theta), t.sin(theta)
```

**Why tensor-typed cos/sin?** If you do `math.cos(theta.item())` you drop the autograd graph and the device — fine for a one-off, broken if `theta` is a learnable parameter or lives on GPU. Always prefer `torch.cos` / `torch.sin` when the input is a tensor.
</details>

### Exercise 2 — build the 3×3 Y-axis rotation matrix

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply the Y-axis rotation matrix formula to assemble a 3×3 tensor from `cos(θ)` and `sin(θ)`.
> Keywords: rotation-matrix, y-axis, tensor-stack
> ```

**KCs targeted:** `rotation-matrix-y-construct`

Implement `ex2_rotation_y(theta)` to return the 3×3 Y-axis rotation matrix as a `(3, 3)` float tensor:

```
R_y(θ) = [[ cos θ,  0,  sin θ],
          [ 0,      1,  0    ],
          [-sin θ,  0,  cos θ]]
```

The middle row is `[0, 1, 0]` because the Y axis is the rotation axis — a point with only a Y component is fixed by the rotation.

Hint: build the matrix via `t.tensor([[c, 0, s], ...])` after extracting `c.item()` / `s.item()`, OR use `t.stack` for autograd-safety. Either is fine for this exercise.

In [ ]:
def ex2_rotation_y(theta: Tensor) -> Tensor:
    """3×3 Y-axis rotation matrix at angle theta."""
    raise NotImplementedError()


def _test_ex2():
    import math
    R = ex2_rotation_y(t.tensor(0.0))
    assert R.shape == (3, 3), f'shape mismatch: {R.shape}'
    assert t.allclose(R, t.eye(3), atol=1e-6), 'R_y(0) should be identity'

    R90 = ex2_rotation_y(t.tensor(math.pi / 2))
    expected = t.tensor([
        [0.0, 0.0, 1.0],
        [0.0, 1.0, 0.0],
        [-1.0, 0.0, 0.0],
    ])
    assert t.allclose(R90, expected, atol=1e-6), f'R_y(pi/2) mismatch:\n{R90}'
    # The middle row MUST be [0, 1, 0] regardless of theta.
    assert t.allclose(R90[1], t.tensor([0.0, 1.0, 0.0])), 'Y row should always be [0, 1, 0]'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_rotation_y(theta: Tensor) -> Tensor:
    c = t.cos(theta)
    s = t.sin(theta)
    return t.tensor([
        [c.item(), 0.0,  s.item()],
        [0.0,      1.0,  0.0],
        [-s.item(), 0.0, c.item()],
    ])
```

**Sign convention.** The matrix above is the standard right-hand rule rotation: looking *down* the +Y axis (from above), the rotation goes counter-clockwise. Some texts (and graphics APIs like DirectX) use the left-hand variant with the signs flipped — `[[c, 0, -s], ..., [s, 0, c]]`. ARENA uses the right-hand convention.

**Autograd-safe alternative.** If `theta` is a learnable parameter, use `torch.stack` to avoid breaking the graph:
```python
row0 = t.stack([c, t.zeros_like(c), s])
row1 = t.tensor([0.0, 1.0, 0.0])
row2 = t.stack([-s, t.zeros_like(c), c])
return t.stack([row0, row1, row2])
```
</details>

### Exercise 3 — rotate a single 3-vector

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply a rotation matrix to a 3-vector via matrix-vector multiplication.
> Keywords: matvec, right-hand-rule, geometric-check
> ```

**KCs targeted:** `rotation-applied-to-vector`

Implement `ex3_rotate_vector(v, theta)` to rotate the 3-vector `v` by angle `theta` about the Y axis. Output shape `(3,)`.

Use `R_y(θ) @ v` where you build `R_y` from Exercise 2.

Geometric check (right-hand rule, looking down +Y):
- `(1, 0, 0)` rotated by π/2 → `(0, 0, -1)` (X goes to -Z).
- `(0, 0, 1)` rotated by π/2 → `(1, 0, 0)` (+Z goes to +X).
- Any `(0, y, 0)` stays fixed.

In [ ]:
def ex3_rotate_vector(v: Tensor, theta: Tensor) -> Tensor:
    """Rotate v (shape (3,)) about Y by theta. Returns (3,)."""
    raise NotImplementedError()


def _test_ex3():
    import math
    half_pi = t.tensor(math.pi / 2)
    # +X axis rotates to -Z under right-hand rule.
    out = ex3_rotate_vector(t.tensor([1.0, 0.0, 0.0]), half_pi)
    assert out.shape == (3,), f'shape mismatch: {out.shape}'
    assert t.allclose(out, t.tensor([0.0, 0.0, -1.0]), atol=1e-6), f'(1,0,0) @ π/2 should be (0,0,-1), got {out.tolist()}'
    # +Z rotates to +X.
    out2 = ex3_rotate_vector(t.tensor([0.0, 0.0, 1.0]), half_pi)
    assert t.allclose(out2, t.tensor([1.0, 0.0, 0.0]), atol=1e-6), f'(0,0,1) @ π/2 should be (1,0,0), got {out2.tolist()}'
    # Y-component fixed.
    out3 = ex3_rotate_vector(t.tensor([0.0, 7.0, 0.0]), half_pi)
    assert t.allclose(out3, t.tensor([0.0, 7.0, 0.0]), atol=1e-6), 'pure Y vector must be fixed by Y rotation'
    # theta = 0 is identity.
    out4 = ex3_rotate_vector(t.tensor([1.0, 2.0, 3.0]), t.tensor(0.0))
    assert t.allclose(out4, t.tensor([1.0, 2.0, 3.0]), atol=1e-6), 'theta=0 must be identity'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_rotate_vector(v: Tensor, theta: Tensor) -> Tensor:
    c = t.cos(theta).item()
    s = t.sin(theta).item()
    R = t.tensor([
        [c, 0.0, s],
        [0.0, 1.0, 0.0],
        [-s, 0.0, c],
    ])
    return R @ v
```

**Why `R @ v` not `v @ R`?** Conventional rotation matrices act on *column* vectors: `v_rotated = R @ v_column`. If you store vectors as rows (Numpy / PyTorch default), you'd write `v_rotated = v_row @ R.T`. Same math — just transpose the matrix to match your vector layout.

**Length preservation.** Rotation matrices are orthogonal (R @ R.T = I), so `||R @ v|| == ||v||` — a great sanity check during debugging. If your output norm changes, you almost certainly have a sign error or an unnormalised axis.
</details>

### Exercise 4 — verify rotation composition R(α)·R(β) = R(α+β)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply matrix multiplication to demonstrate the group property of rotations about a single axis.
> Keywords: composition, group-property, matmul-check
> ```

**KCs targeted:** `rotation-composes-on-axis`

Implement `ex4_compose_rotations(alpha, beta)` to compute the product `R_y(α) @ R_y(β)` and return it as a `(3, 3)` tensor.

The test then independently checks that the returned matrix equals `R_y(α + β)`. This is the 1-parameter group property: rotations about the same axis add their angles.

Use your Y rotation matrix construction from Exercise 2.

In [ ]:
def ex4_compose_rotations(alpha: Tensor, beta: Tensor) -> Tensor:
    """Return R_y(alpha) @ R_y(beta). Should equal R_y(alpha + beta)."""
    raise NotImplementedError()


def _test_ex4():
    import math
    def R_y(theta):
        c, s = math.cos(theta), math.sin(theta)
        return t.tensor([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])

    alpha = t.tensor(0.3)
    beta = t.tensor(1.2)
    product = ex4_compose_rotations(alpha, beta)
    assert product.shape == (3, 3), f'expected (3,3), got {product.shape}'
    # Independent ground truth: R(α + β).
    expected = R_y(alpha.item() + beta.item())
    assert t.allclose(product, expected, atol=1e-5), f'composition broken:\n{product}\nvs\n{expected}'

    # Symmetric case: α + (-α) = 0 → identity.
    out_zero = ex4_compose_rotations(t.tensor(0.7), t.tensor(-0.7))
    assert t.allclose(out_zero, t.eye(3), atol=1e-5), 'R(α) @ R(-α) must be identity'
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_compose_rotations(alpha: Tensor, beta: Tensor) -> Tensor:
    def R_y(theta):
        c = t.cos(theta).item()
        s = t.sin(theta).item()
        return t.tensor([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])
    return R_y(alpha) @ R_y(beta)
```

**Why this only works on the same axis.** `R_y(α) @ R_y(β) = R_y(α+β)` because rotations about the same axis commute. But `R_y(α) @ R_x(β) ≠ R_x(β) @ R_y(α)` in general — that's the whole point of 3-D orientation being non-commutative. Don't try to compose Euler angles by adding!

**Use of the group property.** Want to pre-compute 60 rotations 6° apart for a turntable? Compute `R_y(6°)` once and matrix-multiply it into a running accumulator. Cheaper than rebuilding from scratch each frame.
</details>

### Exercise 5 — rotate a batch of points around Y

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize matrix construction + matmul-shape arithmetic to rotate a (N, 3) batch of points around the Y axis in one operation.
> Keywords: batch-rotation, matmul-shape, ray-tracing, multi-kc
> ```

**KCs targeted:** `rotation-matrix-y-construct`, `rotation-applied-to-vector`, `rotate-batch-of-points`

Implement `ex5_rotate_batch(points, theta)`. Given a `(N, 3)` batch of 3-D points and a scalar angle `theta`, return a `(N, 3)` batch of rotated points.

Strategy: build `R_y(θ)` of shape `(3, 3)` once, then multiply with the batch. Two equivalent ways:
- `points @ R.T` — points are row vectors; transpose R to match.
- `(R @ points.T).T` — explicit column-vector form.

Pick whichever you find clearer. Both produce identical results.

> ⚠️ **Integrative exercise.** Combines 3 KCs (matrix construction, matvec → batched matmul shape arithmetic, batch rotation). Empirical work (Lohr et al. ITiCSE 2025) shows 3-concept exercises drop to ~40% solvability — expect a step up vs Exercises 1-4.

In [ ]:
def ex5_rotate_batch(points: Tensor, theta: Tensor) -> Tensor:
    """Rotate a (N, 3) batch of points around Y by theta. Returns (N, 3)."""
    raise NotImplementedError()


def _test_ex5():
    import math
    half_pi = t.tensor(math.pi / 2)
    points = t.tensor([
        [1.0, 0.0, 0.0],   # +X axis
        [0.0, 0.0, 1.0],   # +Z axis
        [0.0, 5.0, 0.0],   # pure Y (must be fixed)
        [1.0, 2.0, 0.0],   # XY corner
    ])
    out = ex5_rotate_batch(points, half_pi)
    assert out.shape == (4, 3), f'expected (4,3), got {out.shape}'
    expected = t.tensor([
        [0.0, 0.0, -1.0],  # X → -Z
        [1.0, 0.0, 0.0],   # +Z → +X
        [0.0, 5.0, 0.0],   # Y fixed
        [0.0, 2.0, -1.0],  # X→-Z, Y fixed
    ])
    assert t.allclose(out, expected, atol=1e-6), f'value mismatch:\n{out}\nvs\n{expected}'

    # Length-preserving check on a random batch.
    t.manual_seed(42)
    rnd = t.randn(8, 3)
    norms_before = rnd.pow(2).sum(dim=1).sqrt()
    norms_after = ex5_rotate_batch(rnd, t.tensor(0.7)).pow(2).sum(dim=1).sqrt()
    assert t.allclose(norms_before, norms_after, atol=1e-5), 'rotation must preserve per-row norms'

    # theta=0 identity.
    assert t.allclose(ex5_rotate_batch(points, t.tensor(0.0)), points, atol=1e-6)
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_rotate_batch(points: Tensor, theta: Tensor) -> Tensor:
    c = t.cos(theta).item()
    s = t.sin(theta).item()
    R = t.tensor([
        [c, 0.0, s],
        [0.0, 1.0, 0.0],
        [-s, 0.0, c],
    ])
    return points @ R.T
```

**Why `points @ R.T`?** PyTorch stores batches as `(N, D)` — rows are samples. The rotation formula `v' = R @ v` assumes `v` is a *column* vector. Two ways to reconcile:
- Stack as rows, transpose R: `points_row @ R.T` gives `(N, 3)` directly.
- Stack as cols, then transpose result: `(R @ points.T).T` — same math.

Both compile to the same matmul; pick by readability. `@ R.T` is the idiomatic PyTorch form because it preserves the `(N, D)` row-major layout.

**Where you'll use this.** Camera orbiting, object turntables, constructing per-vertex normals after rotating a mesh, generating rotated test data for invariance checks. Anywhere a scene needs to look at the model from a different angle.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()